In [1]:
"""
RAG Ingestion Pipeline
======================

A production-ready, consolidated script that implements a Retrieval-Augmented
Generation (RAG) ingestion pipeline. The pipeline performs the following steps:

1. PDF parsing with Docling (layout-aware document conversion).
2. Semantic chunking with HybridChunker backed by a HuggingFace tokenizer.
3. Metadata extraction with LangExtract using a Groq-hosted Llama model.
4. Embedding generation with a HuggingFace sentence-transformers model.
5. Qdrant vector storage and querying with Qdrant.

The script is designed to be run as a CLI tool and can also be imported as a
module for programmatic use.

Environment variables:
    GROQ_API_KEY      API key for the Groq inference endpoint.
    EMBEDDING_MODEL   Optional. Defaults to "sentence-transformers/all-MiniLM-L6-v2".
    LLM_MODEL         Optional. Defaults to "llama-3.3-70b-versatile".
    QDRANT_PATH       Optional. Defaults to "./qdrant_data".
    COLLECTION_NAME   Optional. Defaults to "rag_collection".

Requirements (install via pip):
    docling
    docling-hybrid-chunker  # or the package exposing HybridChunker
    langextract
    qdrant-client
    sentence-transformers
    transformers
    python-dotenv
    groq
"""

'\nRAG Ingestion Pipeline\n======================\n\nA production-ready, consolidated script that implements a Retrieval-Augmented\nGeneration (RAG) ingestion pipeline. The pipeline performs the following steps:\n\n1. PDF parsing with Docling (layout-aware document conversion).\n2. Semantic chunking with HybridChunker backed by a HuggingFace tokenizer.\n3. Metadata extraction with LangExtract using a Groq-hosted Llama model.\n4. Embedding generation with a HuggingFace sentence-transformers model.\n5. Qdrant vector storage and querying with Qdrant.\n\nThe script is designed to be run as a CLI tool and can also be imported as a\nmodule for programmatic use.\n\nEnvironment variables:\n    GROQ_API_KEY      API key for the Groq inference endpoint.\n    EMBEDDING_MODEL   Optional. Defaults to "sentence-transformers/all-MiniLM-L6-v2".\n    LLM_MODEL         Optional. Defaults to "llama-3.3-70b-versatile".\n    QDRANT_PATH       Optional. Defaults to "./qdrant_data".\n    COLLECTION_NAME   Op

In [16]:
from __future__ import annotations

import logging
import os
from typing import Optional

from dotenv import load_dotenv

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("RAG Pipeline with Docling, Hybrid Search and Qdrant")

# ---------------------------------------------------------------------------
# Environment loading
# ---------------------------------------------------------------------------
load_dotenv()

GROQ_API_KEY: Optional[str] = os.getenv("GROQ_API_KEY")
EMBEDDING_MODEL_NAME: str = os.getenv(
    "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
)
LLM_MODEL_NAME: str = os.getenv("LLM_MODEL", "llama-3.3-70b-versatile")
QDRANT_URL: str = os.getenv("QDRANT_URL")
QDRANT_API_KEY: str = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME: str = os.getenv("COLLECTION_NAME", "rag_collection")
CHUNKER_TOKENIZER_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"


In [17]:
# ---------------------------------------------------------------------------
# Data containers
# ---------------------------------------------------------------------------
import hashlib
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class Chunk:
    """Represents a semantically meaningful chunk of a document."""

    text: str
    index: int
    source_file: str
    page_numbers: List[int] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[List[float]] = None
    chunk_id: str = field(default_factory=lambda: str(uuid.uuid4()))

    def to_point(self) -> Dict[str, Any]:
        """Convert the chunk into a Qdrant point payload structure."""
        return {
            "id": self.chunk_id,
            "vector": self.embedding,
            "payload": {
                "text": self.text,
                "index": self.index,
                "source_file": self.source_file,
                "page_numbers": self.page_numbers,
                "metadata": self.metadata,
            },
        }
        

In [18]:
# ---------------------------------------------------------------------------
# PDF parsing with Docling
# ---------------------------------------------------------------------------
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from pathlib import Path


class PDFParser:
    """Wraps Docling to convert PDF files into structured document objects."""

    def __init__(self) -> None:
        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = False
        pipeline_options.do_table_structure = False
        pipeline_options.do_code_enrichment = False
        pipeline_options.do_formula_enrichment = False

        self._converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options,
                    backend=PyPdfiumDocumentBackend,
                )
            }
        )
        logging.info("PDFParser initialized (fast text-only mode).")

    def parse(self, pdf_path: Path):
        if not pdf_path.exists():
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        logging.info("Parsing PDF: %s", pdf_path)
        return self._converter.convert(str(pdf_path)).document


In [19]:
# ---------------------------------------------------------------------------
# Semantic chunking with HybridChunker
# ---------------------------------------------------------------------------

from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

MAX_TOKENS = 512


class SemanticChunker:
    """Chunks text using HybridChunker with a HuggingFace tokenizer."""

    def __init__(
        self,
        tokenizer_model: str = EMBEDDING_MODEL_NAME,
        chunk_size: int = 512,
    ) -> None:
        self._hf_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        hf_tokenizer = HuggingFaceTokenizer(
            tokenizer=self._hf_tokenizer,
            max_tokens=chunk_size,
        )
        self._chunker = HybridChunker(
            tokenizer=hf_tokenizer, max_tokens=chunk_size, merge_peers=True
        )
        logging.info(
            f"SemanticChunker initialized (model={tokenizer_model}, chunk_size={chunk_size})."
        )

    def _truncate(self, text: str) -> str:
        """Truncate text to MAX_TOKENS to prevent embedding model errors."""
        tokens = self._hf_tokenizer.encode(text, add_special_tokens=True)
        if len(tokens) <= MAX_TOKENS:
            return text
        return self._hf_tokenizer.decode(tokens[:MAX_TOKENS], skip_special_tokens=True)

    def chunk(self, document, source_file: str) -> list[Chunk]:
        logging.info("Chunking document from %s.", source_file)
        chunks: list[Chunk] = []
        for idx, dl_chunk in tqdm(enumerate(self._chunker.chunk(document)), desc="Chunking", unit="chunk"):
            text = dl_chunk.text if hasattr(dl_chunk, "text") else str(dl_chunk)
            chunks.append(
                Chunk(text=self._truncate(text.strip()), index=idx, source_file=source_file)
            )
        logging.info("Produced %d chunks.", len(chunks))
        return chunks


In [20]:
# ---------------------------------------------------------------------------
# Metadata extraction with Groq/Llama (rate-limit-aware parallel)
# ---------------------------------------------------------------------------
import json
import time
import threading
import concurrent.futures
from groq import Groq
from tqdm import tqdm

class MetadataExtractor:
    """Extracts structured metadata from text chunks using a Groq-hosted Llama model.

    Uses a token-bucket semaphore to stay within Groq's free-tier limits:
      - 30 requests per minute  → max_workers=5 with a 0.5s inter-request delay
      - 12,000 tokens per minute → leave headroom by pacing requests
    """

    _SYSTEM_PROMPT = (
        "You are a metadata extraction assistant. "
        "Return ONLY a JSON object with keys: "
        "summary (string), topics (list), entities (list), keywords (list)."
    )

    def __init__(
        self,
        api_key: Optional[str] = GROQ_API_KEY,
        model: str = LLM_MODEL_NAME,
        max_tokens: int = 512,
        temperature: float = 0.0,
        max_workers: int = 5,          
        requests_per_minute: int = 25, 
    ) -> None:
        if not api_key:
            raise ValueError("GROQ_API_KEY is required for metadata extraction.")
        self._model = model
        self._max_tokens = max_tokens
        self._temperature = temperature
        self._max_workers = max_workers
        self._client = Groq(api_key=api_key)
        self._min_interval = 60.0 / requests_per_minute  # seconds between calls
        self._last_call_time = 0.0
        self._rate_lock = threading.Lock()

        logger.info(
            "MetadataExtractor initialized (model=%s, workers=%d, rpm=%d).",
            model, max_workers, requests_per_minute,
        )

    @staticmethod
    def validate_api_key(api_key) -> None:
        """Validate the API key before starting a batch."""
        if not api_key:
            raise ValueError("GROQ_API_KEY is required for metadata extraction.")

    def _throttled_call(self, chunk: "Chunk") -> dict:
        """Acquire the rate-limit slot, then call the API."""
        with self._rate_lock:
            now = time.monotonic()
            wait = self._min_interval - (now - self._last_call_time)
            if wait > 0:
                time.sleep(wait)
            self._last_call_time = time.monotonic()
        return self._call_api(chunk)

    def _call_api(self, chunk: "Chunk") -> dict:
        """Single API call with no retry logic (Groq client retries automatically)."""
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": self._SYSTEM_PROMPT},
                {"role": "user", "content": chunk.text},
            ],
            max_tokens=self._max_tokens,
            temperature=self._temperature,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)

    def extract(self, chunk: "Chunk") -> Dict[str, Any]:
        """Extract metadata for a single chunk, with throttling."""
        try:
            metadata = self._throttled_call(chunk)
        except Exception as exc:
            logger.warning(
                "Metadata extraction failed for chunk %d: %s.", chunk.index, exc
            )
            metadata = {}

        metadata.setdefault("summary", "")
        metadata.setdefault("topics", [])
        metadata.setdefault("entities", [])
        metadata.setdefault("keywords", [])
        return metadata

    def extract_batch(self, chunks: list["Chunk"]) -> None:
        """Extract metadata for all chunks in parallel, mutating chunk.metadata in place."""
        with concurrent.futures.ThreadPoolExecutor(max_workers=self._max_workers) as executor:
            futures = {executor.submit(self.extract, chunk): chunk for chunk in chunks}
            for future in tqdm(
                concurrent.futures.as_completed(futures),
                total=len(futures),
                desc="Metadata",
                unit="chunk",
            ):
                chunk = futures[future]
                chunk.metadata = future.result()


In [21]:
# ---------------------------------------------------------------------------
# Embedding generation
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer


class Embedder:
    """Generates dense vector embeddings for text chunks."""

    def __init__(self, model_name: str = EMBEDDING_MODEL_NAME) -> None:

        self._model = SentenceTransformer(model_name)
        self._dimension = self._model.get_embedding_dimension()

        logger.info(
            "Embedder initialized (model=%s, dimension=%d).",
            model_name,
            self._dimension,
        )

    @property
    def dimension(self) -> int:
        """Return the embedding dimension."""
        return self._dimension

    def embed(self, texts: List[str]) -> List[List[float]]:
        """Embed a batch of texts.

        Args:
            texts: A list of strings to embed.

        Returns:
            A list of embedding vectors (each a list of floats).
        """
        if not texts:
            return []
        vectors = self._model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,
            batch_size=32,
        )
        return [vec.tolist() for vec in vectors]

In [22]:
# ---------------------------------------------------------------------------
# Qdrant vector store
# ---------------------------------------------------------------------------
from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models


class VectorStore:
    """Manages a Qdrant Cloud collection for storage and retrieval."""

    def __init__(
        self,
        url: str = QDRANT_URL,
        api_key: str = QDRANT_API_KEY,
        collection_name: str = COLLECTION_NAME,
        vector_size: int = 384,
        distance: str = "Cosine",
    ) -> None:
        self._qdrant_models = qdrant_models
        self._collection_name = collection_name
        self._client = QdrantClient(url=url, api_key=api_key)
        self._ensure_collection(vector_size=vector_size, distance=distance)
        
        logger.info(
            "VectorStore initialized (url=%s, collection=%s, dim=%d).",
            url,
            collection_name,
            vector_size,
        )

    def _ensure_collection(self, vector_size: int, distance: str) -> None:
        """Create the collection if it does not already exist."""
        
        collections = self._client.get_collections().collections
        existing_names = {c.name for c in collections}
        
        if self._collection_name not in existing_names:
            self._client.create_collection(
                collection_name=self._collection_name,
                vectors_config=self._qdrant_models.VectorParams(
                    size=vector_size,
                    distance=distance,
                ),
            )
            logger.info("Created Qdrant collection: %s", self._collection_name)

    def upsert_chunks(self, chunks: List[Chunk]) -> None:
        """Upsert a list of chunks (with embeddings) into the collection."""
        if not chunks:
            logger.warning("No chunks provided for upsert.")
            return

        points = [c.to_point() for c in chunks if c.embedding is not None]
        if not points:
            logger.warning("No embedded chunks to upsert.")
            return

        self._client.upsert(
            collection_name=self._collection_name,
            points=points,
            wait=True,
        )
        logger.info("Upserted %d points into %s.", len(points), self._collection_name)

    def query(
        self,
        query_vector: List[float],
        top_k: int = 5,
        filters: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, Any]]:
        """Query the vector store for the nearest neighbors.

        Args:
            query_vector: The embedding vector to search with.
            top_k: Number of results to return.
            filters: Optional payload filters.

        Returns:
            A list of result dictionaries with score and payload.
        """
        must_filters = []
        if filters:
            for key, value in filters.items():
                must_filters.append(
                    self._qdrant_models.FieldCondition(
                        key=key,
                        match=self._qdrant_models.MatchValue(value=value),
                    )
                )

        query_filter = (
            self._qdrant_models.Filter(must=must_filters) if must_filters else None
        )

        results = self._client.query_points(
            collection_name=self._collection_name,
            query=query_vector,
            limit=top_k,
            query_filter=query_filter,
        )

        formatted: List[Dict[str, Any]] = []
        for point in results.points:
            formatted.append(
                {
                    "id": str(point.id),
                    "score": float(point.score),
                    "payload": point.payload or {},
                }
            )
        return formatted


In [23]:
# ---------------------------------------------------------------------------
# Pipeline orchestration
# ---------------------------------------------------------------------------
import logging
import hashlib
import uuid


class RAGPipeline:
    """Orchestrates the full RAG ingestion and querying pipeline."""

    def __init__(
        self,
        chunk_size: int = 512,
        enable_metadata_extraction: bool = True,
    ) -> None:
        self.parser = PDFParser()
        self.chunker = SemanticChunker(chunk_size=chunk_size)
        self.embedder = Embedder()
        self.store = VectorStore(vector_size=self.embedder.dimension)
        self.enable_metadata_extraction = enable_metadata_extraction
        if enable_metadata_extraction:
            MetadataExtractor.validate_api_key(GROQ_API_KEY)
        self.extractor = MetadataExtractor() if enable_metadata_extraction else None

    def _stable_chunk_id(self, chunk: Chunk) -> str:
        digest = hashlib.sha256(
            f"{chunk.source_file}:{chunk.index}:{chunk.text}".encode("utf-8")
        ).hexdigest()
        return str(uuid.UUID(digest[:32].ljust(32, "0")))

    def ingest(self, pdf_path: Path) -> list[Chunk]:
        logging.info("=== Starting ingestion for: %s ===", pdf_path.name)

        document = self.parser.parse(pdf_path)
        chunks = self.chunker.chunk(document, source_file=pdf_path.name)

        if not chunks:
            logging.warning("No chunks produced from %s. Skipping.", pdf_path.name)
            return []

        for chunk in chunks:
            chunk.chunk_id = self._stable_chunk_id(chunk)

        if self.extractor is not None:
            logging.info("Extracting metadata for %d chunks (parallel)...", len(chunks))
            self.extractor.extract_batch(chunks)

        logging.info("Generating embeddings for %d chunks...", len(chunks))
        embeddings = self.embedder.embed([c.text for c in chunks])
        for chunk, emb in zip(chunks, embeddings):
            chunk.embedding = emb

        self.store.upsert_chunks(chunks)
        logging.info("=== Ingestion complete for: %s ===", pdf_path.name)
        return chunks

    def query(
        self, query_text: str, top_k: int = 5, filters: dict | None = None
    ) -> list[dict]:
        logging.info("Querying: %r (top_k=%d)", query_text, top_k)
        query_vector = self.embedder.embed([query_text])[0]
        results = self.store.query(query_vector=query_vector, top_k=top_k, filters=filters)
        logging.info("Retrieved %d results.", len(results))
        return results


In [24]:
# Purpose: Initialize the RAGPipeline orchestrator with custom chunking parameters.
# Prerequisites: The RAGPipeline, PDFParser, SemanticChunker, Embedder, and VectorStore classes must be defined above.

import logging

# Ensure notebook outputs logging info
logging.getLogger().setLevel(logging.INFO)

# Initialize the pipeline
# We enable metadata extraction to utilize the Groq/Llama integration
pipeline = RAGPipeline(
    chunk_size=512,
    enable_metadata_extraction=True
)

print("RAG Pipeline successfully initialized.")


2026-07-06 11:49:40 | INFO     | root | PDFParser initialized (fast text-only mode).
2026-07-06 11:49:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 11:49:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json "HTTP/1.1 200 OK"
2026-07-06 11:49:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 11:49:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-06 11:49:40 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sent

RAG Pipeline successfully initialized.


In [25]:
# Purpose: Execute the end-to-end ingestion process for a specific PDF document.
# Estimated Time: 1-3 minutes depending on document length and API latency.

# Define the path to your target PDF
# Using pathlib.Path as per modern Python 3.11+ standards

target_pdf = Path("./machine-learning-engineer-associate-01.pdf")

if not target_pdf.exists():
    print(
        f"Error: Could not find {target_pdf}. Please ensure the file is in the correct directory."
    )
else:
    print(f"Starting ingestion for {target_pdf.name}...")

    # Run the ingestion pipeline
    ingested_chunks = pipeline.ingest(pdf_path=target_pdf)

    print(
        f"Ingestion complete. Successfully processed and stored {len(ingested_chunks)} chunks."
    )

2026-07-06 11:49:51 | INFO     | root | === Starting ingestion for: machine-learning-engineer-associate-01.pdf ===
2026-07-06 11:49:51 | INFO     | root | Parsing PDF: machine-learning-engineer-associate-01.pdf
2026-07-06 11:49:51 | INFO     | docling.datamodel.document | detected formats: [<InputFormat.PDF: 'pdf'>]
2026-07-06 11:49:51 | INFO     | docling.document_converter | Going to convert document batch...
2026-07-06 11:49:51 | INFO     | docling.document_converter | Initializing pipeline for StandardPdfPipeline with options hash 9ed16645843556d8544034deb3a9c0db
2026-07-06 11:49:51 | INFO     | docling.utils.accelerator_utils | Accelerator device: 'cpu'


Starting ingestion for machine-learning-engineer-associate-01.pdf...


2026-07-06 11:49:51 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-layout-heron/revision/main "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1225.43it/s]
2026-07-06 11:49:53 | INFO     | docling.pipeline.base_pipeline | Processing document machine-learning-engineer-associate-01.pdf
2026-07-06 11:52:10 | INFO     | docling.document_converter | Finished converting document machine-learning-engineer-associate-01.pdf in 138.58 sec.
2026-07-06 11:52:10 | INFO     | root | Chunking document from machine-learning-engineer-associate-01.pdf.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3439 > 512). Running this sequence through the model will result in indexing errors
Chunking: 95chunk [00:00, 648.02chunk/s]
2026-07-06 11:52:12 | INFO     | root | Produced 95 chunks.
2026-07-06 11:52:12 | INFO     | root | Extracting metadata for 95 chunks (parallel).

Ingestion complete. Successfully processed and stored 95 chunks.


In [ ]:
# Purpose: Verify the pipeline by querying the Qdrant vector store using natural language.

# Define a query relevant to the ingested document
query_text = "How Could I be better prepared for the exam?"

print(f"Executing Query: '{query_text}'\n")

# Retrieve the top 3 most semantically relevant chunks
search_results = pipeline.query(
    query_text=query_text,
    top_k=3
)

# Display the results
for i, result in enumerate(search_results, start=1):
    score = result.get("score", 0.0)
    payload = result.get("payload", {})
    metadata = payload.get("metadata", {})
    text_preview = payload.get("text", "")
    
    print(f"--- Result {i} (Similarity Score: {score:.4f}) ---")
    print(f"Source File: {payload.get('source_file', 'Unknown')}")
    
    if metadata.get("title"):
        print(f"Extracted Title: {metadata['title']}")
        
    print(f"Content Preview: {text_preview}\n")

In [ ]:
# ---------------------------------------------------------------------------
# Ingest all PDFs from the ingestion-docs directory
# ---------------------------------------------------------------------------

ingestion_dir = Path("../ingestion-docs")
processed_dir = ingestion_dir / "processed"
processed_dir.mkdir(exist_ok=True)

pdf_files = sorted(ingestion_dir.glob("*.pdf"))

if not pdf_files:
    print(f"No PDF files found in {ingestion_dir.resolve()}")
else:
    print(f"Found {len(pdf_files)} documents to ingest:\n")
    for f in pdf_files:
        print(f"  - {f.name}")
    print()

    total_chunks = 0
    failed = []

    for pdf_path in pdf_files:
        try:
            print(f"[{pdf_files.index(pdf_path) + 1}/{len(pdf_files)}] Ingesting: {pdf_path.name}")
            chunks = pipeline.ingest(pdf_path=pdf_path)
            total_chunks += len(chunks)
            pdf_path.rename(processed_dir / pdf_path.name)
            print(f"  ✓ {len(chunks)} chunks stored → moved to processed/\n")
        except Exception as e:
            print(f"  ✗ Failed: {e}\n")
            failed.append(pdf_path.name)

    print(f"Ingestion complete. {len(pdf_files) - len(failed)}/{len(pdf_files)} documents processed.")
    print(f"Total chunks stored: {total_chunks}")
    if failed:
        print(f"Failed documents: {failed}")
